Load

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import tensorflow as tf
import joblib

from src.modeling import (
    train_test_split_time_series,
    scale_data,
    create_sequences_multi_output,
    build_lstm_multi_output
)

df = pd.read_csv("../data/processed/dataset_final.csv")
df = df.sort_values("date")

df_model = df.drop(columns=["date"])

In [3]:
train, test = train_test_split_time_series(df_model)

In [4]:
train_scaled, test_scaled, scaler = scale_data(train, test)

In [5]:
joblib.dump(scaler, "../model/model_multi/scaler_multi.save")

['../model/model_multi/scaler_multi.save']

In [6]:
window_size = 30

X_train, y_train = create_sequences_multi_output(train_scaled, window_size)
X_test, y_test = create_sequences_multi_output(test_scaled, window_size)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1910, 30, 5)
y_train shape: (1910, 5)


In [7]:
n_features = df_model.shape[1]

model = build_lstm_multi_output(
    input_shape=(window_size, n_features),
    n_features=n_features
)

model.summary()

d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,501 (119.14 KB)

 Trainable params: 30,501 (119.14 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - loss: 0.0300 - val_loss: 0.0391
Epoch 2/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0058 - val_loss: 0.0054
Epoch 3/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0044 - val_loss: 0.0053
Epoch 4/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.0037 - val_loss: 0.0038
Epoch 5/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.0030 - val_loss: 0.0030
Epoch 6/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 7/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0024 - val_loss: 0.0045
Epoch 8/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0023 - val_loss: 0.0027
Epoch 9/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.0022 - val_loss: 0.0018
Epoch 10/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 0.0021 - val_loss: 0.0027
Epoch 11/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0020 - val_loss: 0.0026
Epoch 12/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.0

In [9]:
model.save("../model/model_multi/lstm_multi.keras")

In [10]:
from src.modeling import evaluate_model

y_pred_test = model.predict(X_test)

rmse, mape, r2 = evaluate_model(
    y_test[:, 0],  # IHSG saja
    y_pred_test[:, 0]
)

print("RMSE:", rmse)
print("MAPE:", mape)
print("R2:", r2)

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step
RMSE: 0.030914360805105116
MAPE: 0.027968416605705263
R2: 0.8738440180528413


In [11]:
print("Mean IHSG train:", train["ihsg"].mean())
print("Mean IHSG test:", test["ihsg"].mean())

Mean IHSG train: 5809.390737113402
Mean IHSG test: 7080.000082304527
